# Direct Lyapunov MPC With Frozen Output Disturbance

This notebook is the clean experiment entrypoint for the direct frozen-output-disturbance Lyapunov MPC path. It uses the dedicated target solver in `Lyapunov/frozen_output_disturbance_target.py` and the direct rollout/export helpers in `Lyapunov/direct_lyapunov_mpc.py`.

The notebook keeps controller logic out of cells. The cells below only configure experiments, build the plant/model objects, call the Python helpers, and review saved artifacts.

The saved output and input overview figures now come from the existing shared CSTR plotting helper in `Plotting_fns/mpc_plot_fns.py`, while the direct Lyapunov exporter adds the controller-specific target and Lyapunov diagnostics on top of those shared plots.


## Setup

The main visible switches are:

- `target_mode`: `"unbounded"` or `"bounded"`
- `lyapunov_mode`: `"hard"` or `"soft"`
- prediction and control horizons
- `rho_lyap`, `lyap_eps`, and `slack_penalty`
- scenario length and export toggles


In [ ]:
import os
from pprint import pprint

import numpy as np
import pandas as pd
from IPython.display import Image, display

from TD3Agent.reward_functions import make_reward_fn_mpc_quadratic
from Simulation.mpc import compute_observer_gain
from Simulation.system_functions import PolymerCSTR
from Lyapunov.direct_lyapunov_mpc import (
    build_direct_lyapunov_run_bundle,
    design_direct_lyapunov_mpc_solver,
    run_direct_output_disturbance_lyapunov_mpc,
    save_direct_lyapunov_debug_artifacts,
)
from utils.scaling_helpers import apply_min_max
from utils.td3_helpers import load_and_prepare_system_data

In [ ]:
target_mode = "unbounded"
lyapunov_mode = "hard"

predict_h = 9
cont_h = 3
rho_lyap = 0.98
lyap_eps = 1e-9
slack_penalty = 1e6

n_tests = 2
set_points_len = 400
TEST_CYCLE = [False] * max(int(n_tests), 1)
warm_start = 0

save_debug_bundle = True
save_plots = True
run_comparison_grid = False
comparison_modes = [
    ("unbounded", "hard"),
    ("bounded", "hard"),
    ("unbounded", "soft"),
    ("bounded", "soft"),
]

active_config = {
    "target_mode": target_mode,
    "lyapunov_mode": lyapunov_mode,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "rho_lyap": rho_lyap,
    "lyap_eps": lyap_eps,
    "slack_penalty": slack_penalty,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "save_debug_bundle": save_debug_bundle,
    "save_plots": save_plots,
    "run_comparison_grid": run_comparison_grid,
}

print("Active direct Lyapunov MPC configuration")
pprint(active_config)

## Plant And Controller Setup

In [ ]:
Ad = 2.142e17
Ed = 14897
Ap = 3.816e10
Ep = 3557
At = 4.50e12
Et = 843
fi = 0.6
m_delta_H_r = -6.99e4
hA = 1.05e6
rhocp = 1506
rhoccpc = 4043
Mm = 104.14
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

CIf = 0.5888
CMf = 8.6981
Qi = 108.0
Qs = 459.0
Tf = 330.0
Tcf = 295.0
V = 3000.0
Vc = 3312.4
system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

Qm_ss = 378.0
Qc_ss = 471.6
system_steady_state_inputs = np.array([Qc_ss, Qm_ss])
delta_t = 0.5

steady_states = {"ss_inputs": system_steady_state_inputs.copy()}
cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t, deviation_form=False)
steady_states["y_ss"] = cstr_ss.y_ss.copy()

u_min = np.array([71.6, 78.0])
u_max = np.array([870.0, 670.0])
setpoint_y_phys = np.array([[4.5, 324.0], [3.4, 321.0]])

system_data = load_and_prepare_system_data(
    steady_states=steady_states,
    setpoint_y=setpoint_y_phys,
    u_min=u_min,
    u_max=u_max,
    system_dict_path=os.path.join("Data", "system_dict"),
    augmentation_style="rawlings",
    augmentation_mode="output_disturbance",
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]

inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(setpoint_y_phys, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(
    steady_states["y_ss"],
    data_min[inputs_number:],
    data_max[inputs_number:],
)

poles = np.array([
    0.44619852,
    0.33547649,
    0.36380595,
    0.70467118,
    0.3562966,
    0.42900673,
    0.4228262,
    0.96916776,
    0.91230187,
])
L = compute_observer_gain(A_aug, C_aug, poles)

u_ss_scaled = apply_min_max(steady_states["ss_inputs"], data_min[:inputs_number], data_max[:inputs_number])
u_min_scaled = apply_min_max(u_min, data_min[:inputs_number], data_max[:inputs_number])
u_max_scaled = apply_min_max(u_max, data_min[:inputs_number], data_max[:inputs_number])

u_dev_min = u_min_scaled - u_ss_scaled
u_dev_max = u_max_scaled - u_ss_scaled
bnds = tuple((float(lo), float(hi)) for lo, hi in zip(u_dev_min, u_dev_max)) * cont_h
IC_opt = np.zeros(inputs_number * cont_h)

Qy_diag = np.array([5.0, 1.0])
Su_diag = np.array([1.0, 1.0])
Rdu_diag = np.array([1.0, 1.0])

reward_config, reward_fn = make_reward_fn_mpc_quadratic(Q_diag=Qy_diag, R_diag=Rdu_diag)

LMPC_obj = design_direct_lyapunov_mpc_solver(
    A_aug=A_aug,
    B_aug=B_aug,
    C_aug=C_aug,
    Qy_diag=Qy_diag,
    NP=predict_h,
    NC=cont_h,
    Su_diag=Su_diag,
    u_min=u_dev_min,
    u_max=u_dev_max,
    Rdu_diag=Rdu_diag,
    terminal_set_on=True,
    terminal_alpha_scale=1.0,
    terminal_cost_scale=1.0,
)

nominal_qs = 459.0
nominal_qi = 108.0
nominal_hA = 1.05e6
qi_change = 0.95
qs_change = 1.05
ha_change = 0.92

def run_case(case_target_mode, case_lyapunov_mode, save_artifacts=False, save_case_plots=False):
    case_config = {
        **active_config,
        "target_mode": case_target_mode,
        "lyapunov_mode": case_lyapunov_mode,
    }
    print(f"Running case target_mode={case_target_mode}, lyapunov_mode={case_lyapunov_mode}")
    pprint(case_config)

    cstr_case = PolymerCSTR(
        system_params,
        system_design_params,
        system_steady_state_inputs,
        delta_t,
        deviation_form=False,
    )
    results_case = run_direct_output_disturbance_lyapunov_mpc(
        system=cstr_case,
        LMPC_obj=LMPC_obj,
        y_sp_scenario=y_sp_scenario,
        n_tests=n_tests,
        set_points_len=set_points_len,
        steady_states=steady_states,
        IC_opt=IC_opt,
        bnds=bnds,
        L=L,
        data_min=data_min,
        data_max=data_max,
        test_cycle=TEST_CYCLE,
        reward_fn=reward_fn,
        nominal_qi=nominal_qi,
        nominal_qs=nominal_qs,
        nominal_ha=nominal_hA,
        qi_change=qi_change,
        qs_change=qs_change,
        ha_change=ha_change,
        target_mode=case_target_mode,
        lyapunov_mode=case_lyapunov_mode,
        target_config={},
        target_H=None,
        mode="disturb",
        disturbance_after_step=True,
        use_target_output_for_tracking=False,
        skip_terminal_if_alpha_small=True,
        alpha_terminal_min=1e-8,
        use_target_on_solver_fail=False,
        rho_lyap=rho_lyap,
        lyap_eps=lyap_eps,
        slack_penalty=slack_penalty,
        first_step_contraction_on=True,
        reset_system_on_entry=True,
        solver_options={"warm_start": True},
    )
    prefix_name = f"direct_lyapunov_mpc_{case_target_mode}_{case_lyapunov_mode}"
    bundle_case = build_direct_lyapunov_run_bundle(
        source=prefix_name,
        results=results_case,
        steady_states=steady_states,
        config=case_config,
        data_min=data_min,
        data_max=data_max,
        extra={"reward_config": reward_config, "min_max_dict": min_max_dict},
    )
    debug_dir_case = None
    if save_artifacts:
        debug_dir_case = save_direct_lyapunov_debug_artifacts(
            bundle_case,
            directory=os.path.join(os.getcwd(), "Data", "debug_exports"),
            prefix_name=prefix_name,
            save_plots=save_case_plots,
            save_paper_plots=save_case_plots,
        )
    return results_case, bundle_case, debug_dir_case

## Single Run

In [ ]:
results, bundle, debug_dir = run_case(
    target_mode,
    lyapunov_mode,
    save_artifacts=save_debug_bundle,
    save_case_plots=save_plots,
)

summary_df = pd.DataFrame([bundle["summary"]])
summary_df

## Comparison Sweep

In [ ]:
comparison_df = None
if run_comparison_grid:
    comparison_rows = []
    for case_target_mode, case_lyapunov_mode in comparison_modes:
        _, bundle_case, _ = run_case(
            case_target_mode,
            case_lyapunov_mode,
            save_artifacts=False,
            save_case_plots=False,
        )
        comparison_rows.append(bundle_case["summary"])
    comparison_df = pd.DataFrame(comparison_rows)
    comparison_df = comparison_df[
        [
            "target_mode",
            "lyapunov_mode",
            "reward_mean",
            "solver_success_rate",
            "target_success_rate",
            "hard_contraction_rate",
            "relaxed_contraction_rate",
            "slack_lyap_mean",
            "slack_lyap_max",
            "bounded_solution_used_steps",
            "target_residual_total_norm_max",
        ]
    ]
comparison_df

## Saved Artifacts

The saved plot directory now contains two plot groups:

- shared CSTR MPC figures from `Plotting_fns/mpc_plot_fns.py`
- direct Lyapunov diagnostics from `Lyapunov/direct_lyapunov_mpc.py`


In [ ]:
if save_debug_bundle:
    print(debug_dir)
    plot_dir = None if debug_dir is None else os.path.join(debug_dir, "plots")
    shared_plot_names = [
        "fig_mpc_outputs_full.png",
        f"fig_mpc_outputs_last{bundle['time_in_sub_episodes']}.png",
        "fig_mpc_inputs_full.png",
        f"fig_mpc_inputs_last{bundle['time_in_sub_episodes']}.png",
    ]
    direct_plot_names = [
        "01_outputs_vs_targets.png",
        "02_inputs_vs_targets.png",
        "03_state_target_error.png",
        "04_lyapunov_diagnostics.png",
        "05_target_diagnostics.png",
        "06_tail_window_summary.png",
    ]
    if plot_dir and os.path.isdir(plot_dir):
        print("Shared MPC plot views")
        for name in shared_plot_names:
            candidate = os.path.join(plot_dir, name)
            if os.path.exists(candidate):
                display(Image(filename=candidate))
        print("Direct Lyapunov diagnostics")
        for name in direct_plot_names:
            candidate = os.path.join(plot_dir, name)
            if os.path.exists(candidate):
                display(Image(filename=candidate))